In [ ]:
import sys
import traceback
import logging
import portalocker
import os

# 경로 설정 - 플랫폼에 따라 다르게 처리
if os.name == 'nt':  # Windows
    sys.path.insert(0, r"Y:/git/pyaedt_library/src/")
else:  # Linux/Unix
    # Linux 서버 경로들 시도
    possible_paths = [
        # r"/gpfs/home1/r1jae262/jupyter/git/pyaedt_library/src/",
        r"../pyaedt_library/src/",
        os.path.join(os.path.dirname(__file__), "../git/pyaedt_library/src/"),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            sys.path.insert(0, path)
            break

import pyaedt_module
from pyaedt_module.core import pyDesktop
import os
import time
from datetime import datetime

import math
import copy

import pandas as pd


import platform
import csv



from module.input_parameter import create_input_parameter, create_input_parameter_for_test, calculate_coil_parameter, calculate_coil_offset, set_design_variables
from module.modeling import (
    create_core_model, create_all_windings, create_cold_plate, create_air,
    assign_meshing, assign_excitations, create_face, create_mold
)
from module.report import (
    get_input_parameter, get_maxwell_magnetic_parameter,
    get_maxwell_calculator_parameter, get_convergence_report, get_icepak_calculator_parameter
)



class Simulation() :

    def __init__(self, desktop=None) :

        self.NUM_CORE = 4
        self.NUM_TASK = 1

        # Desktop은 바깥에서 생성(with 포함)해서 주입하는 것을 권장
        # (루프 중 프로젝트 close/delete가 Desktop 핸들을 무효화시키는 문제를 줄이기 위함)
        self.desktop = desktop

    def create_simulation_name(self):

        file_path = "./simulation_num.txt"

        # 파일이 존재하지 않으면 생성
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as file:
                file.write("1")

        # 읽기/쓰기 모드로 파일 열기
        with open(file_path, "r+", encoding="utf-8") as file:
            # 파일 잠금: LOCK_EX는 배타적 잠금,  블로킹 모드로 실행 (Windows/Linux 호환)
            portalocker.lock(file, portalocker.LOCK_EX)

            # 파일에서 값 읽기
            content = int(file.read().strip())
            self.num = content
            self.PROJECT_NAME = f"simulation{content}"
            content += 1

            # 파일 포인터를 처음으로 되돌리고, 파일 내용 초기화 후 새 값 쓰기
            file.seek(0)
            file.truncate()
            file.write(str(content))

    def create_input_parameter(self, param_list=None) :
        if self.test == True :
            input_parameter = create_input_parameter_for_test(self.maxwell_design, param_list)
        else :
            input_parameter = create_input_parameter(self.maxwell_design, param_list)
        logging.info(f"input_parameter : {input_parameter}")
        logging.info("input_parameter : " + ",".join(str(float(v)) for v in input_parameter.values()))
        return input_parameter

    def set_variable(self, input_parameter) :
        # 1. Simulation 클래스 인스턴스에 속성으로 값을 설정합니다 (예: self.N1 = 10).
        for key, value in input_parameter.items():
            setattr(self.maxwell_design, key, value)
        
        self.input_df = pd.DataFrame([input_parameter])
        
        # 2. input_parameter.py의 함수를 호출하여 Ansys 디자인에 변수를 설정합니다.
        set_design_variables(self.maxwell_design, input_parameter)

    def set_maxwell_analysis(self) :
        self.maxwell_design.setup = self.maxwell_design.create_setup(name = "Setup1")
        self.maxwell_design.setup.properties["Max. Number of Passes"] = 12 # 10
        self.maxwell_design.setup.properties["Min. Number of Passes"] = 1
        self.maxwell_design.setup.properties["Min. Converged Passes"] = 3
        self.maxwell_design.setup.properties["Percent Error"] = 2.5 # 2.5
        self.maxwell_design.setup.properties["Frequency Setup"] = f"{self.maxwell_design.frequency}kHz"

    def create_geometry(self, design):

        core_params = {
            "w1": "300mm", # 변수가 아니라 값을 넣어도 됨
            "l1_leg": "50mm",
            "l1_top": "50mm",
            "l2": "200mm",
            "h1": "300mm",
            "mat": "ferrite"
        }

        core = design.modeler.create_coretype_core(name="Core", **core_params)



def run(simulation=None):
    """시뮬레이션을 실행합니다."""
    sim1 = simulation

    sim1.create_simulation_name()

    # simulation 디렉토리 생성 (존재하지 않으면)
    simulation_dir = "./simulation"
    if not os.path.exists(simulation_dir):
        os.makedirs(simulation_dir, exist_ok=True)
    
    # 절대 경로로 변환
    project_path = os.path.abspath(os.path.join(simulation_dir, sim1.PROJECT_NAME))
    
    # desktop이 None이거나 유효하지 않은지 확인
    if sim1.desktop is None:
        raise RuntimeError("Desktop instance is None. Cannot create project.")
    
    try:
        project1 = sim1.desktop.create_project(path=project_path, name=sim1.PROJECT_NAME)
    except Exception as e:
        error_msg = f"Failed to create project '{sim1.PROJECT_NAME}' at path '{project_path}': {e}\n"
        print(error_msg, file=sys.stderr)
        sys.stderr.flush()
        raise
    
    design1 = project1.create_design(name="mawell_design", solver="maxwell3d", solution="AC Magnetic")

    # input_parameter = sim1.create_input_parameter()
    # sim1.set_variable(input_parameter)

    sim1.project = project1
    sim1.design1 = design1



itr = 0
GUI = False
desktop = pyDesktop(version=None, non_graphical=GUI, close_on_exit=False, new_desktop=True)
sim = Simulation(desktop=desktop)
run(simulation=sim)

"""
itr = 0
GUI = False
with pyDesktop(version=None, non_graphical=GUI, close_on_exit=False, new_desktop=True) as desktop:
    print(f"loop {itr} : desktop init done (pid={getattr(desktop, 'pid', None)})", flush=True)
    print(f"loop {itr} : simulation start!!", flush=True)
    sim = Simulation(desktop=desktop)
    run(simulation=sim)
"""

In [ ]:

def create_coil(name, window_height, window_length, window_layer, N_input, width_fill_factor, height_fill_factor, space_length, space_width, shape="circle", offset = [0,0,0]):

    # name : coil의 name
    # window_height : 권선 윈도우의 높이
    # 
    coil_width = 0
    coil_height = 0
    coil_gap_x = 0
    coil_gap_z = 0

    if N_input != None :
        N = N_input

    shape = shape.lower()

    if shape == "circle" :

        if window_layer > 1 :
            coil_width = window_length * width_fill_factor / window_layer
            coil_height = coil_width
        elif window_layer == 1 :
            coil_width = window_length
            coil_height = coil_width
    

        if window_layer > 1 :
            coil_gap_x = (window_length - (coil_width * window_layer)) / (window_layer - 1)
        else :
            coil_gap_x = 0

        N = int(window_height / coil_width * height_fill_factor)
        coil_gap_z = (window_height - (coil_width * N)) / (N - 1)



    elif shape == "rectangle" :

        coil_width = window_length * width_fill_factor / window_layer
        coil_height = window_height * height_fill_factor / N

        if window_layer > 1 :
            coil_gap_x = (window_length - (coil_width * window_layer)) / (window_layer - 1)
        else :
            coil_gap_x = 0

        if N > 1 :
            coil_gap_z = (window_height - (coil_height * N)) / (N - 1)
        else :
            coil_gap_z = 0


    x_pos = []
    y_pos = []
    z_pos = []

    for i in range(window_layer) :
        x_pos.append(space_length/2 + coil_width*(i+0.5) + coil_gap_x*i)
        y_pos.append(space_width/2 + coil_width*(i+0.5) + coil_gap_x*i)

    for i in range(N) :
        z_pos.append(window_height/2 - coil_height*(i+0.5) - coil_gap_z*i)

    windings = []

    i = 0
    j = 0

    for i, (x, y) in enumerate(zip(x_pos, y_pos)):
        for j, z in enumerate(z_pos):
                
            x_pos_s = x
            x_neg_s = -x
            y_pos_s = y
            y_neg_s = -y

            points = []
            points.append([f"{x_pos_s}mm + {offset[0]}mm", f"{y_pos_s}mm + {offset[1]}mm", f"{z}mm + {offset[2]}mm" ])
            points.append([f"{x_neg_s}mm + {offset[0]}mm", f"{y_pos_s}mm + {offset[1]}mm", f"{z}mm + {offset[2]}mm"])
            points.append([f"{x_neg_s}mm + {offset[0]}mm", f"{y_neg_s}mm + {offset[1]}mm", f"{z}mm + {offset[2]}mm"])
            points.append([f"{x_pos_s}mm + {offset[0]}mm", f"{y_neg_s}mm + {offset[1]}mm", f"{z}mm + {offset[2]}mm"])
            points.append([f"{x_pos_s}mm + {offset[0]}mm", f"{y_pos_s}mm + {offset[1]}mm", f"{z}mm + {offset[2]}mm" ])


            winding = sim.design1.modeler.create_polyline(
                points=points, name=f"{name}_{i}_{j}", xsection_orient="Auto",
                xsection_type=shape, xsection_width=coil_width, xsection_height=coil_height, xsection_num_seg=6, xsection_topwidth=coil_width)
            windings.append(winding)

    print(f"N = {N}")
    print(f"coil_width = {coil_width}")
    print(f"coil_height = {coil_height}")
    print(f"coil_gap_x = {coil_gap_x}")
    print(f"coil_gap_z = {coil_gap_z}")

    return windings, N, coil_width, coil_height,coil_gap_x, coil_gap_z



def create_core(name, l1, l2, h1, w1, core_material="ferrite") :

    main = sim.design1.modeler.create_box(
        origin = [f"-{(4*l1+2*l2)/2}mm", f"-{(w1)/2}mm", f"-{(h1+2*l1)/2}mm"],
        sizes = [f"{4*l1+2*l2}mm", f"{w1}mm", f"{h1+2*l1}mm"],
        name = f"{name}",
        material = core_material
    )

    sub1 = sim.design1.modeler.create_box(
        origin = [f"-{l1}mm", f"-{(w1)/2}mm", f"-{(h1)/2}mm"],
        sizes = [f"-{l2}mm", f"{w1}mm", f"{h1}mm"],
        name = f"{name}_sub1",
        material = core_material
    )

    sub2 = sim.design1.modeler.create_box(
        origin = [f"{l1}mm", f"-{(w1)/2}mm", f"-{(h1)/2}mm"],
        sizes = [f"{l2}mm", f"{w1}mm", f"{h1}mm"],          
        name = f"{name}_sub2",
        material = core_material
    )

    
    sim.design1.modeler.subtract(
        [main],
        [sub1, sub2],
        keep_originals=False
    )

    return main







In [ ]:




# 카탈로그 손실점(B, f, P[W/kg]) 저장 위치
# - POSCO GO/NO: 공식 카탈로그 손실점
# - M19/M27: 대표 문헌값 + 기존 하드코딩 계수 출처 명시
CORE_LOSS_BFP_POINTS = {
    "POSCO_23PH085_GO": [(1.7, 50.0, 0.85)],
    "POSCO_27PG130_GO": [(1.7, 50.0, 1.30)],
    "POSCO_35PN210_NO": [(1.5, 50.0, 2.10)],
    "POSCO_20PNX1150F_HYPER_NO": [(1.0, 400.0, 11.5)],
    "POSCO_25PNX1250F": [(1.0, 400.0, 12.5)],
    "POSCO_27PNF1500": [(1.0, 400.0, 15.0)],
    "M19_STD": [(1.0, 50.0, 1.72), (1.5, 50.0, 4.00), (1.0, 60.0, 2.07), (1.5, 60.0, 4.81)],
    "M27_STD": [(1.0, 50.0, 2.16), (1.5, 50.0, 5.08), (1.0, 60.0, 2.60), (1.5, 60.0, 6.10)],
    "JFE_6P5SI_JNEX": [],
}

# Typical electrical/magnetic properties 표에서 읽은 점을 여기에 추가
# 형식: (B[T], f[Hz], P[W/kg])
CORE_LOSS_BFP_TYPICAL_POINTS = {
    "POSCO_23PH085_GO": [],
    "POSCO_27PG130_GO": [],
    "POSCO_35PN210_NO": [],
    # 20PNX1150F: PNX1150F catalog page의 Typical table + iron loss curve 추정치
    # source: material/PNX1150F.png
    "POSCO_20PNX1150F_HYPER_NO": [
        (0.2, 50.0, 0.04), (0.4, 50.0, 0.15), (0.6, 50.0, 0.35), (0.8, 50.0, 0.62), (1.0, 50.0, 0.95), (1.5, 50.0, 2.01),
        (0.2, 200.0, 0.22), (0.4, 200.0, 0.85), (0.6, 200.0, 1.8), (0.8, 200.0, 3.3), (1.0, 200.0, 5.1),
        (0.2, 400.0, 0.55), (0.4, 400.0, 2.0), (0.6, 400.0, 4.5), (0.8, 400.0, 7.8), (1.0, 400.0, 10.6),
        (0.2, 800.0, 1.4), (0.4, 800.0, 6.5), (0.6, 800.0, 12.0), (0.8, 800.0, 21.0), (1.0, 800.0, 27.6),
        (0.2, 1000.0, 2.0), (0.4, 1000.0, 7.8), (0.6, 1000.0, 16.5), (0.8, 1000.0, 28.0), (1.0, 1000.0, 38.1),
    ],
    # source: material/posco_6p5%Si.pdf (PNX Core 25PNX1250F iron loss curve + typical table)
    "POSCO_25PNX1250F": [
        (0.2, 50.0, 0.05), (0.4, 50.0, 0.18), (0.6, 50.0, 0.38), (0.8, 50.0, 0.65), (1.0, 50.0, 0.95), (1.5, 50.0, 1.97),
        (0.2, 200.0, 0.28), (0.4, 200.0, 1.05), (0.6, 200.0, 2.25), (0.8, 200.0, 4.00), (1.0, 200.0, 6.20),
        (0.2, 400.0, 0.75), (0.4, 400.0, 2.80), (0.6, 400.0, 5.80), (0.8, 400.0, 10.20), (1.0, 400.0, 12.1),
        (0.2, 800.0, 2.20), (0.4, 800.0, 7.80), (0.6, 800.0, 16.50), (0.8, 800.0, 28.50), (1.0, 800.0, 33.9),
        (0.2, 1000.0, 3.30), (0.4, 1000.0, 11.50), (0.6, 1000.0, 23.50), (0.8, 1000.0, 40.50), (1.0, 1000.0, 47.7),
    ],
    # source: material/posco_6p5%Si.pdf (PNF Core 27PNF1500 iron loss curve + typical table)
    "POSCO_27PNF1500": [
        (0.2, 50.0, 0.06), (0.4, 50.0, 0.22), (0.6, 50.0, 0.45), (0.8, 50.0, 0.78), (1.0, 50.0, 1.15), (1.5, 50.0, 2.14),
        (0.2, 200.0, 0.33), (0.4, 200.0, 1.25), (0.6, 200.0, 2.60), (0.8, 200.0, 4.60), (1.0, 200.0, 7.20),
        (0.2, 400.0, 0.85), (0.4, 400.0, 3.20), (0.6, 400.0, 6.70), (0.8, 400.0, 11.80), (1.0, 400.0, 13.2),
        (0.2, 800.0, 2.40), (0.4, 800.0, 8.80), (0.6, 800.0, 18.50), (0.8, 800.0, 32.00), (1.0, 800.0, 36.8),
        (0.2, 1000.0, 3.60), (0.4, 1000.0, 13.00), (0.6, 1000.0, 27.00), (0.8, 1000.0, 46.50), (1.0, 1000.0, 51.3),
    ],
    "JFE_6P5SI_JNEX": [],
    "M19_STD": [],
    "M27_STD": [],
}

# 반자동 digitize 결과가 있으면 PNX1150F typical 포인트를 자동 치환
try:
    import json as _json
    import os as _os

    _digitized_path = "material/pnx1150f_digitized_points.json"
    if _os.path.exists(_digitized_path):
        with open(_digitized_path, "r", encoding="utf-8") as _f:
            _dig = _json.load(_f)

        _freq_map = {"50Hz": 50.0, "200Hz": 200.0, "400Hz": 400.0, "800Hz": 800.0, "1000Hz": 1000.0}
        _pts = []
        _by_freq = {}
        for _freq, _arr in _dig.get("points_by_frequency", {}).items():
            _hz = _freq_map.get(_freq)
            if _hz is None:
                continue
            _pairs = []
            for _b, _p in _arr:
                _b = float(_b)
                _p = float(_p)
                _pairs.append((_b, _p))
                _pts.append((_b, float(_hz), _p))
            _pairs.sort(key=lambda t: t[0])
            _by_freq[_freq] = _pairs

        def _mono_ok(_pairs):
            if not _pairs:
                return False
            _vals = [p for _, p in _pairs]
            return all(_vals[i + 1] >= _vals[i] for i in range(len(_vals) - 1))

        _valid = True

        # 핵심 시각 검증 포인트: 800Hz @ 0.4T 는 대략 6~7W/kg 범위
        _p_800_04 = None
        for _b, _p in _by_freq.get("800Hz", []):
            if abs(_b - 0.4) < 1e-9:
                _p_800_04 = _p
                break
        if _p_800_04 is None or not (6.0 <= _p_800_04 <= 7.0):
            _valid = False
            print(f"[CORE_LOSS_DIGITIZED] invalid 800Hz@0.4T={_p_800_04}, expected 6~7")

        for _f in ("50Hz", "200Hz", "400Hz", "800Hz", "1000Hz"):
            if not _mono_ok(_by_freq.get(_f, [])):
                _valid = False
                print(f"[CORE_LOSS_DIGITIZED] non-monotonic curve: {_f}")

        if _pts and _valid:
            CORE_LOSS_BFP_TYPICAL_POINTS["POSCO_20PNX1150F_HYPER_NO"] = _pts
            print(f"[CORE_LOSS_DIGITIZED] loaded {len(_pts)} points from {_digitized_path}")
        else:
            print("[CORE_LOSS_DIGITIZED] rejected, fallback to built-in typical points")
except Exception as _e:
    print(f"[CORE_LOSS_DIGITIZED] skipped: {_e}")

# 다중 material digitize 결과가 있으면 해당 material의 typical 포인트를 자동 치환
try:
    import json as _json
    import os as _os

    _multi_path = "material/posco_curves_digitized.json"
    if _os.path.exists(_multi_path):
        with open(_multi_path, "r", encoding="utf-8") as _f:
            _multi = _json.load(_f)

        for _mat, _obj in _multi.get("materials", {}).items():
            _pts = []
            for _b, _hz, _p in _obj.get("points_bfp", []):
                _pts.append((float(_b), float(_hz), float(_p)))
            if _pts and _mat in CORE_LOSS_BFP_TYPICAL_POINTS:
                CORE_LOSS_BFP_TYPICAL_POINTS[_mat] = _pts
                print(f"[CORE_LOSS_DIGITIZED_MULTI] {_mat}: {len(_pts)} points loaded")
except Exception as _e:
    print(f"[CORE_LOSS_DIGITIZED_MULTI] skipped: {_e}")

# core loss 데이터 사용 모드
# - typical: typical/curve 데이터 우선(없으면 spec_max fallback)
# - spec_max: 규격 최대손실점만 사용
# - combined: 둘 다 합쳐 사용
core_loss_data_mode = "typical"

SILICON_STEEL_PRESETS = {
    "POSCO_23PH085_GO": {"base": "steel_1008", "density": 7650, "kh": None, "kc": None, "ke": None, "loss_spec": "W17/50=0.85", "source": "https://www.posco.co.kr/homepage/product/common/s91pdownload.jsp?file=/hfiles/productpr/71f7d1ec184ac0c1138a61a30ebdc510.pdf"},
    "POSCO_27PG130_GO": {"base": "steel_1008", "density": 7650, "kh": None, "kc": None, "ke": None, "loss_spec": "W17/50=1.30", "source": "https://www.posco.co.kr/homepage/product/common/s91pdownload.jsp?file=/hfiles/productpr/71f7d1ec184ac0c1138a61a30ebdc510.pdf"},
    "POSCO_35PN210_NO": {"base": "steel_1008", "density": 7650, "kh": None, "kc": None, "ke": None, "loss_spec": "W15/50=2.10", "source": "https://www.posco.co.kr/homepage/product/common/s91pdownload.jsp?file=/hfiles/productpr/71f7d1ec184ac12a083a61a30ebdc510.pdf"},
    "POSCO_20PNX1150F_HYPER_NO": {"base": "steel_1008", "density": 7650, "kh": None, "kc": None, "ke": None, "loss_spec": "W10/400=11.5", "source": "https://www.posco.co.kr/homepage/product/common/s91pdownload.jsp?file=/hfiles/productpr/61eee5fd18bcce75a7ce861c8173b413.pdf"},
    "POSCO_25PNX1250F": {"base": "steel_1008", "density": 7600, "kh": None, "kc": None, "ke": None, "loss_spec": "W10/400(max)=12.5, typical=12.1", "source": "material/posco_6p5%Si.pdf (PNX Core page)"},
    "POSCO_27PNF1500": {"base": "steel_1008", "density": 7600, "kh": None, "kc": None, "ke": None, "loss_spec": "W10/400(max)=15.0, typical=13.2", "source": "material/posco_6p5%Si.pdf (PNF Core page)"},

    # 6.5% Si 계열: 실리콘스틸이 맞음(고실리콘 NO), 일반 3%급과 적용 주파수 대역이 다름
    "JFE_6P5SI_JNEX": {"base": "steel_1008", "density": 7650, "kh": None, "kc": None, "ke": None, "loss_spec": "datasheet points required", "source": "https://www.jfe-steel.co.jp/en/products/electrical/product/supercore/index.php"},

    # 하드코딩 계수 + 출처 명시
    "M19_STD": {"base": "steel_1008", "density": 7650, "kh": 0.0275, "kc": 1.83e-5, "ke": 2.5e-4, "source": "https://raw.githubusercontent.com/ansys/pyaedt/main/src/ansys/aedt/core/modules/material.py ; ASTM A677 representative fit"},
    "M27_STD": {"base": "steel_1008", "density": 7650, "kh": 0.0330, "kc": 2.10e-5, "ke": 3.00e-4, "source": "https://raw.githubusercontent.com/ansys/pyaedt/main/src/ansys/aedt/core/modules/material.py ; ASTM A677 representative fit"},
}

selected_core_steel = "POSCO_20PNX1150F_HYPER_NO"

def _select_bfp_points(preset_key, mode="typical"):
    spec_points = list(CORE_LOSS_BFP_POINTS.get(preset_key, []))
    typical_points = list(CORE_LOSS_BFP_TYPICAL_POINTS.get(preset_key, []))

    if mode == "spec_max":
        return spec_points
    if mode == "typical":
        return typical_points if typical_points else spec_points
    return spec_points + typical_points

def _fit_or_scale_khkc_ke(points, reference=(0.0275, 1.83e-5, 2.5e-4)):
    # Maxwell Electrical Steel model basis: P = kh*f*B^2 + kc*f^2*B^2 + ke*(f*B)^1.5
    def basis(B, f):
        return [f * (B ** 2), (f ** 2) * (B ** 2), (f * B) ** 1.5]

    def solve_3x3(a, b):
        m = [a[0][:] + [b[0]], a[1][:] + [b[1]], a[2][:] + [b[2]]]
        for k in range(3):
            pivot_row = max(range(k, 3), key=lambda r: abs(m[r][k]))
            if abs(m[pivot_row][k]) < 1e-18:
                return None
            if pivot_row != k:
                m[k], m[pivot_row] = m[pivot_row], m[k]
            pivot = m[k][k]
            for c in range(k, 4):
                m[k][c] /= pivot
            for r in range(3):
                if r == k:
                    continue
                fac = m[r][k]
                for c in range(k, 4):
                    m[r][c] -= fac * m[k][c]
        return m[0][3], m[1][3], m[2][3]

    # 일반 least-squares + ridge regularization
    # - 다중 주파수 + 3점 이상: 거의 순수 fitting
    # - 단일 주파수/점 부족: reference 비율을 유지하는 regularized fitting
    if len(points) >= 2:
        freqs = {float(f) for _, f, _ in points}
        multifreq = len(freqs) >= 2
        lam = 1e-10 if (multifreq and len(points) >= 3) else 1e-2

        # least squares via normal equations (3x3)
        ata = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
        atp = [0.0, 0.0, 0.0]
        for B, f, P in points:
            v = basis(float(B), float(f))
            for i in range(3):
                atp[i] += v[i] * float(P)
                for j in range(3):
                    ata[i][j] += v[i] * v[j]

        # ridge term: (A^T A + lam I)x = A^T P + lam x_ref
        for i in range(3):
            ata[i][i] += lam
            atp[i] += lam * float(reference[i])

        solved = solve_3x3(ata, atp)
        if solved is not None:
            kh, kc, ke = solved
            if kh > 0 and kc > 0 and ke > 0:
                mode = "fit_multifreq" if (multifreq and len(points) >= 3) else "fit_regularized"
                return kh, kc, ke, mode

    # 비정상 해일 때: 기준계수 비례 스케일(전체 포인트 기반)
    if not points:
        raise ValueError("No B-f-P points available for coefficient estimation")

    scales = []
    for B0, f0, P0 in points:
        b0 = basis(float(B0), float(f0))
        pref = reference[0] * b0[0] + reference[1] * b0[1] + reference[2] * b0[2]
        if pref > 0:
            scales.append(float(P0) / pref)

    if not scales:
        raise ValueError("Invalid reference model for coefficient scaling")

    scales.sort()
    scale = scales[len(scales) // 2]
    mode = "scaled_from_multi_point" if len(scales) > 1 else "scaled_from_single_point"
    return reference[0] * scale, reference[1] * scale, reference[2] * scale, mode

def ensure_silicon_steel_material(design, preset_key):
    preset = SILICON_STEEL_PRESETS[preset_key]
    mat_name = f"silicon_steel_{preset_key.lower()}"

    mat = None
    try:
        if mat_name.lower() in [m.lower() for m in design.materials.mat_names]:
            mat = design.materials[mat_name]
        else:
            mat = design.materials.duplicate_material(preset["base"], mat_name)
            if not mat:
                mat = design.materials.add_material(mat_name)
    except Exception:
        mat = design.materials.add_material(mat_name)

    if mat is None:
        raise RuntimeError(f"Failed to create material preset: {preset_key}")

    mat.mass_density = preset["density"]

    fit_mode = "preset"
    if any(preset.get(k) is None for k in ("kh", "kc", "ke")):
        points = _select_bfp_points(preset_key, core_loss_data_mode)
        print(f"[CORE_LOSS_POINTS] mode={core_loss_data_mode} count={len(points)}")
        kh, kc, ke, fit_mode = _fit_or_scale_khkc_ke(points)
        preset["kh"], preset["kc"], preset["ke"] = kh, kc, ke

    if hasattr(mat, "set_electrical_steel_coreloss"):
        mat.set_electrical_steel_coreloss(
            kh=preset["kh"],
            kc=preset["kc"],
            ke=preset["ke"],
            kdc=0,
            cut_depth="1mm",
        )

    print(f"[CORE_LOSS_FIT] mode={fit_mode} kh={preset['kh']:.6g} kc={preset['kc']:.6g} ke={preset['ke']:.6g}")

    return mat_name



core_material_name = ensure_silicon_steel_material(sim.design1, selected_core_steel)
print(f"[CORE_MATERIAL] {selected_core_steel} -> {core_material_name}")
print(f"[CORE_MATERIAL_SOURCE] {SILICON_STEEL_PRESETS[selected_core_steel].get('source')}")

In [ ]:


l1 = 75
l2 = 250
h1 = 650
w1 = 500





# core = create_core(name="core", l1=l1, l2=l2, h1=h1, w1=w1, core_material=core_material_name)
core = create_core(name="core", l1=l1, l2=l2, h1=h1, w1=w1, core_material="ferrite")


In [ ]:
win1_core_space = 40
win2_core_space = 40
win1_win2_space = 30
win2_win2_space = 20


net_window_length = l2 - win1_core_space - win2_core_space - win1_win2_space - win2_win2_space

ratio = 0.35 # space를 빼고 남은 공간 중 Tx가 차지할 비율

Tx_window_length = net_window_length * ratio
Rx_window_length = net_window_length - Tx_window_length 

print(f"Tx_window_length: {Tx_window_length}, Rx_window_length: {Rx_window_length}")

49+91+40+40+30

In [ ]:

window_height = (h1 - 2*win1_core_space) * 0.6
window_length = Tx_window_length
window_layer = 1
window_fill_factor = 1.0
N_fill_factor = 0.75
space_length = 2*l1 + 2*win1_core_space
space_width = w1 + 2*win1_core_space

Tx_windings, Tx_N, Tx_coil_width, Tx_coil_height, Tx_coil_gap_x, Tx_coil_gap_z = create_coil("Tx", window_height, window_length, window_layer, 5, window_fill_factor, N_fill_factor, space_length, space_width, "circle", offset=[0,0,0])

for winding in Tx_windings :

    winding.transparency = 0
    winding.color = [255, 10, 10]
    winding.material_name = "copper"

In [ ]:
window_layer1 = 15 # side
window_layer2 = 60 - window_layer1 # center

# window_layer1 = 1
# window_layer2 = 1


window_height = h1 - 2*win2_core_space

window_length1 = Rx_window_length * (window_layer1) / (window_layer1 + window_layer2)
window_length2 = Rx_window_length * (window_layer2) / (window_layer1 + window_layer2)

window_fill_factor = 0.5
N_fill_factor = 1.0

space_length1 = l1 + 2*win2_core_space
space_width1 = w1 + 2*win2_core_space

space_length2 = 2*l1 + 2*win2_core_space + 2*win1_win2_space + 2*Tx_window_length
space_width2 = w1 + 2*win2_core_space + 2*win1_win2_space + 2*Tx_window_length

# window_fill_factor = 1


# window_height = h1 - 2*coil_coil_space
# window_length = Rx_window_length
# window_layer = 10
# window_fill_factor = 0.65
# N_fill_factor = 1.0
# space_length = l1 + 2*coil_coil_space
# space_width = w1 + 2*coil_coil_space

Rx_windings1, Rx_N1, Rx_coil_width1, Rx_coil_height1, Rx_coil_gap_x1, Rx_coil_gap_z1 = create_coil("Rx1", window_height, window_length1, window_layer1, 1, window_fill_factor, N_fill_factor, space_length1, space_width1, "rectangle", offset=[(-l1-l2-l1/2),0,0])
Rx_windings2, Rx_N2, Rx_coil_width2, Rx_coil_height2, Rx_coil_gap_x2, Rx_coil_gap_z2 = create_coil("Rx2", window_height, window_length1, window_layer1, 1, window_fill_factor, N_fill_factor, space_length1, space_width1, "rectangle", offset=[(l1+l2+l1/2),0,0])


Rx_windings3, Rx_N3, Rx_coil_width3, Rx_coil_height3, Rx_coil_gap_x3, Rx_coil_gap_z3 = create_coil("Rx3", window_height, window_length2, window_layer2, 1, window_fill_factor, N_fill_factor, space_length2, space_width2, "rectangle", offset=[0,0,0])


for winding in Rx_windings1 + Rx_windings2 + Rx_windings3 :

    winding.transparency = 0
    winding.color = [10, 10, 255]
    winding.material_name = "copper"


In [ ]:
def section_winding_get_two_sheets(sim, winding_obj, sheet_prefix=None, plane="ZX"):
    """
    plane="ZX"  -> global y=0 평면(=XZ plane)
    returns: (sheets, ok)
      - sheets: 새로 생성된 sheet object 리스트
      - ok: section 성공 여부(bool)
    """
    modeler = sim.design1.modeler

    if sheet_prefix is None:
        sheet_prefix = f"{winding_obj.name}_sec"

    # section 전 sheet 목록 저장
    before_sheet_names = set(modeler.sheet_names)

    # PyAEDT section 실행 (returns bool)
    ok = modeler.section(winding_obj, plane)  # plane choices: "XY","YZ","ZX"  :contentReference[oaicite:1]{index=1}
    if not ok:
        return [], False

    # section 후 새로 생긴 sheet 이름들
    after_sheet_names = set(modeler.sheet_names)
    new_sheet_names = list(after_sheet_names - before_sheet_names)

    # sheet object로 변환
    sheets = [modeler.get_object_from_name(n) for n in new_sheet_names]

    # face seperation
    sheets = sim.design1.modeler.separate_bodies(assignment=sheets)



    # (옵션) 딱 2개만 기대한다면, winding bbox 근처만 남기고 싶을 수 있음
    # 여기선 일단 다 반환
    return sheets, True


Tx_sheets_pos = []
Tx_sheets_neg = []
Rx_sheets1_pos = []
Rx_sheets1_neg = []
Rx_sheets2_pos = []
Rx_sheets2_neg = []
Rx_sheets3_pos = []
Rx_sheets3_neg = []

for winding in Tx_windings:
    sheets, ok = section_winding_get_two_sheets(sim, winding)
    if ok == True and len(sheets) == 2:
        sheet0, sheet1 = sheets[0], sheets[1]
        # 각각의 sheet의 중심을 비교해서 큰 쪽을 pos, 작은 쪽을 neg로 구분
        center0 = sheet0.faces[0].center[0]
        center1 = sheet1.faces[0].center[0]
        if center0 > center1:
            Tx_sheets_pos.append(sheet0)
            Tx_sheets_neg.append(sheet1)
        else:
            Tx_sheets_pos.append(sheet1)
            Tx_sheets_neg.append(sheet0)

for winding in Rx_windings1:
    sheets, ok = section_winding_get_two_sheets(sim, winding)
    if ok == True and len(sheets) == 2:
        sheet0, sheet1 = sheets[0], sheets[1]
        # 각각의 sheet의 중심을 비교해서 큰 쪽을 pos, 작은 쪽을 neg로 구분
        center0 = sheet0.faces[0].center[0]
        center1 = sheet1.faces[0].center[0]
        if center0 > center1:
            Rx_sheets1_pos.append(sheet0)
            Rx_sheets1_neg.append(sheet1)
        else:
            Rx_sheets1_pos.append(sheet1)
            Rx_sheets1_neg.append(sheet0)

for winding in Rx_windings2:
    sheets, ok = section_winding_get_two_sheets(sim, winding)
    if ok == True and len(sheets) == 2:
        sheet0, sheet1 = sheets[0], sheets[1]
        # 각각의 sheet의 중심을 비교해서 큰 쪽을 pos, 작은 쪽을 neg로 구분
        center0 = sheet0.faces[0].center[0]
        center1 = sheet1.faces[0].center[0]
        if center0 > center1:
            Rx_sheets2_pos.append(sheet0)
            Rx_sheets2_neg.append(sheet1)
        else:
            Rx_sheets2_pos.append(sheet1)
            Rx_sheets2_neg.append(sheet0)

for winding in Rx_windings3:
    sheets, ok = section_winding_get_two_sheets(sim, winding)
    if ok == True and len(sheets) == 2:
        sheet0, sheet1 = sheets[0], sheets[1]
        # 각각의 sheet의 중심을 비교해서 큰 쪽을 pos, 작은 쪽을 neg로 구분
        center0 = sheet0.faces[0].center[0]
        center1 = sheet1.faces[0].center[0]
        if center0 > center1:
            Rx_sheets3_pos.append(sheet0)
            Rx_sheets3_neg.append(sheet1)
        else:
            Rx_sheets3_pos.append(sheet1)
            Rx_sheets3_neg.append(sheet0)





In [ ]:
tx_winding = sim.design1.assign_winding(
    assignment=[], 
    winding_type="Current", 
    is_solid=True, 
    current=f"{1000*math.sqrt(2)}A",
    name="Tx_winding"
)
rx_winding1 = sim.design1.assign_winding(
    assignment=[], 
    winding_type="Current", 
    is_solid=True, 
    current=f"{100*math.sqrt(2)}A",
    name="Rx_winding"
)

In [ ]:
import re

Tx_coil_pos = []
Tx_coil_neg = []
Rx_coil_pos = []
Rx_coil_neg = []



for sheet in Tx_sheets_neg :

    match = re.match(r'^(Tx_\d+_\d+).*$', sheet.name)
    if match:
        new_name = f"n_{match.group(1)}_face"
        sheet.name = new_name
    coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity="Positive", name=f"{sheet.name}_coil")
    Tx_coil_neg.append(coil)


for sheet in (Rx_sheets1_neg + Rx_sheets2_neg) :

    match = re.match(r'^(Rx[12]_\d+_\d+).*$', sheet.name)
    if match:
        new_name = f"n_{match.group(1)}_face"
        sheet.name = new_name
    coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity=f"Positive", name=f"{sheet.name}_coil")
    Rx_coil_neg.append(coil)

for sheet in Rx_sheets3_neg :

    match = re.match(r'^(Rx3_\d+_\d+).*$', sheet.name)
    if match:
        new_name = f"n_{match.group(1)}_face"
        sheet.name = new_name
    coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity=f"Negative", name=f"{sheet.name}_coil")
    Rx_coil_neg.append(coil)



sim.design1.add_winding_coils(assignment="Tx_winding", coils=[coil.name for coil in Tx_coil_neg])
sim.design1.add_winding_coils(assignment="Rx_winding", coils=[coil.name for coil in Rx_coil_neg])

sim.design1.assign_matrix(matrix_name="Matrix", assignment=["Tx_winding", "Rx_winding"])



In [ ]:

"""
length_mesh = sim.design1.mesh.assign_length_mesh(
    assignment=[core],
    inside_selection=False,
    maximum_length="100mm",
    name="core_mesh"
)

"""

In [ ]:
"""
freq = 1e+3

mu0 = 4 * math.pi * 1e-7
mu_copper = mu0 
sigma_copper = 58000000
omega = 2 * math.pi * freq
skin_depth = math.sqrt(2 / (omega * mu_copper * sigma_copper)) * 1e3 # in mm


Tx_skin_depth_mesh = sim.design1.mesh.assign_skin_depth(
    assignment=Tx_windings,
    skin_depth=f'{skin_depth}mm',
    triangulation_max_length='50mm',
    layers_number="2",
    name="Tx_winding_skin_depth"
)

Rx_skin_depth_mesh = sim.design1.mesh.assign_skin_depth(
    assignment=Rx_windings1 + Rx_windings2 + Rx_windings3,
    skin_depth=f'{skin_depth}mm',
    triangulation_max_length='50mm',
    layers_number="2",
    name="Rx_winding_skin_depth"
)
"""


In [ ]:
air_region = sim.design1.modeler.create_air_region(x_pos=100.0, y_pos=100.0, z_pos=100.0, x_neg=100.0, y_neg=100.0, z_neg=100.0, is_percentage=True)
sim.design1.assign_radiation(assignment=[air_region.name], radiation="Radiation")

In [ ]:
sim.design1.setup = sim.design1.create_setup(name = "Setup1")
sim.design1.setup.properties["Max. Number of Passes"] = 6 # 10
sim.design1.setup.properties["Min. Number of Passes"] = 1
sim.design1.setup.properties["Min. Converged Passes"] = 1
sim.design1.setup.properties["Percent Error"] = 2.5 # 2.5
sim.design1.setup.properties["Frequency Setup"] = f"1kHz"


In [ ]:
sim.design1.setup.analyze(cores=16)

In [ ]:
params = [
    ["Matrix.L(Tx_winding,Tx_winding)", f"Ltx", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)", f"Lrx", "uH"],
    ["Matrix.L(Tx_winding,Rx_winding)", f"M", "uH"],
    ["abs(Matrix.CplCoef(Tx_winding,Rx_winding))", f"k", ""],
    ["Matrix.L(Tx_winding,Tx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmr", "uH"],
    ["Matrix.L(Tx_winding,Tx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llr", "uH"],
]

dir = sim.project.path
mod = "write"
import_report = None
report_name = "magnetic_report"
file_name = "magnetic_report"

report, df = sim.design1.get_magnetic_parameter(dir=dir, parameters=params, mod=mod, import_report=import_report, report_name=report_name, file_name=file_name)